# BHC Faster R-CNN — Kaggle + Roboflow

This notebook uses the Roboflow YOLOv8 dataset download and trains the existing Faster R-CNN pipeline for **150 epochs**.

Roboflow project: `human-coastal-zmajf`, version 1.


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
!pip install -q roboflow torchmetrics pycocotools albumentations


In [ ]:
from roboflow import Roboflow
import os

rf = Roboflow(api_key="XslFaDHQ2fu2uFnuAF5K")
project = rf.workspace("image-analysis-project-rfnyq").project("human-coastal-zmajf")
version = project.version(1)
dataset = version.download("yolov8")

LOCAL_DATASET_PATH = dataset.location
DATASET_PATH = LOCAL_DATASET_PATH
DATASET_DRIVE_PATH = LOCAL_DATASET_PATH
print("Dataset downloaded to:", LOCAL_DATASET_PATH)
print(os.listdir(LOCAL_DATASET_PATH))


In [ ]:
import yaml

with open(os.path.join(LOCAL_DATASET_PATH, "data.yaml")) as f:
    data_yaml = yaml.safe_load(f)

print(data_yaml)
CLASS_NAMES_RAW = {i: name for i, name in enumerate(data_yaml['names'])}
print("Classes:", CLASS_NAMES_RAW)

In [ ]:
from PIL import Image
import torch
from torch.utils.data import Dataset
import torchvision.transforms.functional as F

RAW_TO_MODEL = {0: 1, 1: 2}   # torchvision reserves label 0 for background
MODEL_TO_RAW = {1: 0, 2: 1}
NUM_CLASSES_WITH_BG = 3

class YoloDetectionDataset(Dataset):
    def __init__(self, split_dir, transforms=None):
        self.images_dir = os.path.join(split_dir, "images")
        self.labels_dir = os.path.join(split_dir, "labels")
        self.transforms = transforms
        self.image_files = sorted([
            f for f in os.listdir(self.images_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.images_dir, img_name)
        label_path = os.path.join(self.labels_dir, os.path.splitext(img_name)[0] + ".txt")

        img = Image.open(img_path).convert("RGB")
        img_w, img_h = img.size

        boxes, labels = [], []
        if os.path.exists(label_path):
            with open(label_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    cls, xc, yc, w, h = parts
                    cls = int(cls)
                    xc, yc, w, h = float(xc), float(yc), float(w), float(h)

                    x1 = (xc - w / 2) * img_w
                    y1 = (yc - h / 2) * img_h
                    x2 = (xc + w / 2) * img_w
                    y2 = (yc + h / 2) * img_h

                    if x2 <= x1 or y2 <= y1:
                        continue

                    boxes.append([x1, y1, x2, y2])
                    labels.append(RAW_TO_MODEL[cls])

        boxes = torch.as_tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4), dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64) if labels else torch.zeros((0,), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]) if len(boxes) else torch.zeros((0,)),
            "iscrowd": torch.zeros((len(boxes),), dtype=torch.int64),
        }

        img = F.to_tensor(img)
        if self.transforms:
            img, target = self.transforms(img, target)
        return img, target

def collate_fn(batch):
    return tuple(zip(*batch))

In [ ]:
from torch.utils.data import DataLoader

TRAIN_DIR = os.path.join(LOCAL_DATASET_PATH, "train")
VALID_DIR = os.path.join(LOCAL_DATASET_PATH, "valid")
TEST_DIR  = os.path.join(LOCAL_DATASET_PATH, "test")

train_dataset = YoloDetectionDataset(TRAIN_DIR)
valid_dataset = YoloDetectionDataset(VALID_DIR)
test_dataset  = YoloDetectionDataset(TEST_DIR)

print("Train:", len(train_dataset), " Valid:", len(valid_dataset), " Test:", len(test_dataset))

BATCH_SIZE = 4
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, collate_fn=collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, collate_fn=collate_fn)

# sanity check
img, target = train_dataset[0]
print(img.shape, target)

In [ ]:
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

def get_model(num_classes=NUM_CLASSES_WITH_BG):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

model = get_model()
model.to(device)

In [ ]:
import time

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.1)

NUM_EPOCHS = 150
history = {"train_loss": [], "val_loss": [], "loss_classifier": [], "loss_box_reg": [], "loss_objectness": [], "loss_rpn_box_reg": []}

def evaluate_loss(model, data_loader):
    model.train()
    total_loss, n = 0.0, 0
    with torch.no_grad():
        for images, targets in data_loader:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            loss_dict = model(images, targets)
            total_loss += sum(loss for loss in loss_dict.values()).item()
            n += 1
    return total_loss / max(n, 1)

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0.0
    epoch_components = {"loss_classifier": 0.0, "loss_box_reg": 0.0, "loss_objectness": 0.0, "loss_rpn_box_reg": 0.0}
    t0 = time.time()

    for images, targets in train_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        epoch_loss += losses.item()
        for k in epoch_components:
            if k in loss_dict:
                epoch_components[k] += loss_dict[k].item()

    lr_scheduler.step()
    avg_train_loss = epoch_loss / len(train_loader)
    avg_val_loss = evaluate_loss(model, valid_loader)

    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    for k in epoch_components:
        history[k].append(epoch_components[k] / len(train_loader))

    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] train_loss={avg_train_loss:.4f} val_loss={avg_val_loss:.4f} time={time.time()-t0:.1f}s")

CKPT_PATH = os.path.join(DATASET_DRIVE_PATH, "fasterrcnn_beach_human.pth")
torch.save(model.state_dict(), CKPT_PATH)
print("Saved:", CKPT_PATH)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(history["train_loss"], label="Train Loss", marker='o')
plt.plot(history["val_loss"], label="Validation Loss", marker='o')
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Faster R-CNN Training vs Validation Loss")
plt.legend(); plt.grid(alpha=0.3)
plt.savefig("/content/loss_curve.png", dpi=150, bbox_inches='tight')
plt.show()

plt.figure(figsize=(8,5))
for k in ["loss_classifier", "loss_box_reg", "loss_objectness", "loss_rpn_box_reg"]:
    plt.plot(history[k], label=k, marker='o')
plt.xlabel("Epoch"); plt.ylabel("Loss component"); plt.title("Loss Component Breakdown per Epoch")
plt.legend(); plt.grid(alpha=0.3)
plt.savefig("/content/loss_components.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
model.eval()
all_preds, all_targets = [], []

with torch.no_grad():
    for images, targets in test_loader:
        images = [img.to(device) for img in images]
        outputs = model(images)
        preds = [{k: v.cpu() for k, v in o.items()} for o in outputs]
        tgts  = [{k: v.cpu() for k, v in t.items()} for t in targets]
        all_preds.extend(preds)
        all_targets.extend(tgts)

print("Collected predictions for", len(all_preds), "test images")

In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision

metric = MeanAveragePrecision(class_metrics=True)
metric.update(all_preds, all_targets)
results = metric.compute()

print("mAP@[0.5:0.95]:", round(results['map'].item(), 4))
print("mAP@0.5:        ", round(results['map_50'].item(), 4))
print("mAP@0.75:       ", round(results['map_75'].item(), 4))
print("mAR (100 dets): ", round(results['mar_100'].item(), 4))

if 'map_per_class' in results and results['map_per_class'].numel() > 0:
    classes = results.get('classes', torch.tensor([1, 2])).tolist()
    for model_label, ap in zip(classes, results['map_per_class'].tolist()):
        raw = MODEL_TO_RAW.get(model_label, model_label)
        print(f"  AP class {raw} ({CLASS_NAMES_RAW.get(raw, raw)}): {round(ap,4)}")

In [ ]:
import numpy as np

def box_iou(box1, box2):
    x1, y1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    x2, y2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0

IOU_THRESH = 0.5
SCORE_THRESH = 0.5

y_true, y_pred, y_scores = [], [], []
total_gt, total_matched, total_fp = 0, 0, 0

for pred, tgt in zip(all_preds, all_targets):
    matched_gt = set()
    p_boxes, p_labels, p_scores = pred['boxes'].numpy(), pred['labels'].numpy(), pred['scores'].numpy()
    g_boxes, g_labels = tgt['boxes'].numpy(), tgt['labels'].numpy()
    total_gt += len(g_boxes)

    keep = p_scores > SCORE_THRESH
    p_boxes, p_labels, p_scores = p_boxes[keep], p_labels[keep], p_scores[keep]

    for pb, pl, ps in zip(p_boxes, p_labels, p_scores):
        best_iou, best_j = 0, -1
        for j, gb in enumerate(g_boxes):
            if j in matched_gt:
                continue
            iou = box_iou(pb, gb)
            if iou > best_iou:
                best_iou, best_j = iou, j
        if best_iou >= IOU_THRESH:
            matched_gt.add(best_j)
            total_matched += 1
            y_true.append(MODEL_TO_RAW.get(int(g_labels[best_j]), int(g_labels[best_j])))
            y_pred.append(MODEL_TO_RAW.get(int(pl), int(pl)))
            y_scores.append(ps)
        else:
            total_fp += 1  # unmatched prediction = false positive

y_true, y_pred, y_scores = np.array(y_true), np.array(y_pred), np.array(y_scores)
print(f"Total GT boxes: {total_gt} | Matched (TP): {total_matched} | Unmatched preds (FP): {total_fp}")

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              classification_report, confusion_matrix, ConfusionMatrixDisplay)

detection_accuracy = total_matched / total_gt if total_gt > 0 else 0
classification_accuracy = accuracy_score(y_true, y_pred)  # accuracy of 0-vs-1 label among matched detections

precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)

print(f"Detection recall (TP / total GT boxes): {detection_accuracy:.4f}")
print(f"Classification accuracy (0 vs 1, on matched boxes): {classification_accuracy:.4f}")
print(f"Precision (macro): {precision_macro:.4f}")
print(f"Recall (macro):    {recall_macro:.4f}")
print(f"F1-score (macro):  {f1_macro:.4f}")
print()
print(classification_report(y_true, y_pred, target_names=[CLASS_NAMES_RAW.get(0,"class0"), CLASS_NAMES_RAW.get(1,"class1")], zero_division=0))

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[f"0: {CLASS_NAMES_RAW.get(0,'class0')}", f"1: {CLASS_NAMES_RAW.get(1,'class1')}"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix — Faster R-CNN")
plt.savefig("/content/confusion_matrix.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

plt.figure(figsize=(7,6))
for cls, name in CLASS_NAMES_RAW.items():
    binary_true = (y_true == cls).astype(int)
    binary_scores = np.where(y_pred == cls, y_scores, 0)
    if binary_true.sum() == 0:
        continue
    precision, recall, _ = precision_recall_curve(binary_true, binary_scores)
    ap = average_precision_score(binary_true, binary_scores)
    plt.plot(recall, precision, label=f"{name} (AP={ap:.3f})")

plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("Precision-Recall Curve per Class")
plt.legend(); plt.grid(alpha=0.3)
plt.savefig("/content/pr_curve.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
metrics_summary = {
    "Precision": precision_macro,
    "Recall": recall_macro,
    "F1-score": f1_macro,
    "mAP@0.5": results['map_50'].item(),
    "mAP@[.5:.95]": results['map'].item(),
}

plt.figure(figsize=(8,5))
bars = plt.bar(metrics_summary.keys(), metrics_summary.values(), color=['#4C72B0','#55A868','#C44E52','#8172B2','#CCB974'])
plt.ylim(0, 1.0)
plt.ylabel("Score")
plt.title("Faster R-CNN — Summary Metrics")
for bar, val in zip(bars, metrics_summary.values()):
    plt.text(bar.get_x() + bar.get_width()/2, val + 0.02, f"{val:.3f}", ha='center')
plt.grid(axis='y', alpha=0.3)
plt.savefig("/content/metrics_summary.png", dpi=150, bbox_inches='tight')
plt.show()

print(metrics_summary)

In [ ]:
import matplotlib.patches as patches

def show_predictions(dataset, model, num_images=4, score_thresh=0.6):
    model.eval()
    fig, axes = plt.subplots(1, num_images, figsize=(5*num_images, 5))
    for i in range(num_images):
        img, target = dataset[i]
        with torch.no_grad():
            pred = model([img.to(device)])[0]
        img_np = img.permute(1,2,0).numpy()
        ax = axes[i] if num_images > 1 else axes
        ax.imshow(img_np)
        for box, label, score in zip(pred['boxes'], pred['labels'], pred['scores']):
            if score < score_thresh:
                continue
            raw_label = MODEL_TO_RAW.get(int(label), int(label))
            x1,y1,x2,y2 = box.cpu().numpy()
            rect = patches.Rectangle((x1,y1), x2-x1, y2-y1, linewidth=2, edgecolor='lime', facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y1-5, f"{raw_label}:{score:.2f}", color='lime', fontsize=10, weight='bold')
        ax.axis('off')
    plt.tight_layout()
    plt.savefig("/content/sample_predictions.png", dpi=150, bbox_inches='tight')
    plt.show()

show_predictions(test_dataset, model, num_images=4)

In [ ]:
import time

model.eval()
sample_imgs = [test_dataset[i][0].to(device) for i in range(min(20, len(test_dataset)))]

# warm-up
with torch.no_grad():
    _ = model([sample_imgs[0]])

times = []
with torch.no_grad():
    for img in sample_imgs:
        t0 = time.time()
        _ = model([img])
        times.append(time.time() - t0)

avg_inference_time = np.mean(times)
fps = 1 / avg_inference_time
print(f"Avg inference time per image: {avg_inference_time*1000:.1f} ms")
print(f"Approx FPS: {fps:.2f}")

In [ ]:
RESULTS_DIR = os.path.join(DATASET_DRIVE_PATH, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

for fname in ["loss_curve.png", "loss_components.png", "confusion_matrix.png",
              "pr_curve.png", "metrics_summary.png", "sample_predictions.png"]:
    src = f"/content/{fname}"
    if os.path.exists(src):
        shutil.copy(src, os.path.join(RESULTS_DIR, fname))

import json
final_results = {
    "mAP_0.5": results['map_50'].item(),
    "mAP_0.5:0.95": results['map'].item(),
    "mAR_100": results['mar_100'].item(),
    "precision_macro": precision_macro,
    "recall_macro": recall_macro,
    "f1_macro": f1_macro,
    "classification_accuracy": classification_accuracy,
    "avg_inference_time_ms": avg_inference_time * 1000,
    "fps": fps,
}
with open(os.path.join(RESULTS_DIR, "final_results.json"), "w") as f:
    json.dump(final_results, f, indent=2)

print("Saved to:", RESULTS_DIR)
print(final_results)

In [ ]:
import os
import torch
import matplotlib.pyplot as plt
from torchvision.transforms.functional import to_pil_image
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# ADDED for NameError: name 'test_dataset' is not defined and FileNotFoundError
from PIL import Image
from torch.utils.data import Dataset
import torchvision.transforms.functional as F
import shutil, glob, zipfile # Added for dataset re-extraction logic

# Folder in Google Drive
SAVE_DIR = "/kaggle/working/Faster_RCNN_Beach_Results/test_predictions"
os.makedirs(SAVE_DIR, exist_ok=True)

# Number of test images to save
NUM_IMAGES = 10

# Confidence threshold
SCORE_THRESH = 0.5

# --- Fix for NameError: name 'model' is not defined (from previous edit) ---
# This block ensures 'model' and its immediate dependencies are defined if the kernel state was lost.

# Re-define 'device' if it's not in the global scope (from cell 4E6Hs67ARA7p)
if 'device' not in globals():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Warning: 'device' was not defined. Re-initialized to {device}.")

# Re-define 'NUM_CLASSES_WITH_BG' if it's not in the global scope (from cell yBpYcLb3YkyV)
if 'NUM_CLASSES_WITH_BG' not in globals():
    NUM_CLASSES_WITH_BG = 3 # This value is constant in the notebook
    print(f"Warning: 'NUM_CLASSES_WITH_BG' was not defined. Re-initialized to {NUM_CLASSES_WITH_BG}.")

# Re-define 'DATASET_DRIVE_PATH' if it's not in the global scope (from cell -2_8jq-EVSA5)
if 'DATASET_DRIVE_PATH' not in globals():
    # Assuming the path from previous notebook execution
    DATASET_DRIVE_PATH = "LOCAL_DATASET_PATH"
    print(f"Warning: 'DATASET_DRIVE_PATH' was not defined. Re-initialized to {DATASET_DRIVE_PATH}.")

# Define 'get_model' function and instantiate/load 'model' if it's not in the global scope (from cell TTP8aTIyZE9b and tyuFffBiZIms)
if 'model' not in globals():
    print("Warning: 'model' was not defined. Initializing and loading weights.")

    def get_model(num_classes=NUM_CLASSES_WITH_BG):
        model_instance = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(weights="DEFAULT")
        in_features = model_instance.roi_heads.box_predictor.cls_score.in_features
        model_instance.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
        return model_instance

    model = get_model()

    CKPT_PATH = os.path.join(DATASET_DRIVE_PATH, "fasterrcnn_beach_human.pth")
    if os.path.exists(CKPT_PATH):
        model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
        print(f"Loaded model weights from {CKPT_PATH}")
    else:
        print(f"Error: Model checkpoint not found at {CKPT_PATH}. The model will use default, untrained weights.")

    model.to(device)

# ADDED: Re-initialize MODEL_TO_RAW and CLASS_NAMES_RAW for display if they are not defined
if 'MODEL_TO_RAW' not in globals():
    MODEL_TO_RAW = {1: 0, 2: 1} # Assuming these values from yBpYcLb3YkyV
    print("Warning: 'MODEL_TO_RAW' was not defined. Re-initialized.")
if 'CLASS_NAMES_RAW' not in globals():
    CLASS_NAMES_RAW = {0: '0', 1: '1'} # Assuming values from 7bRF-5uoWJDC
    print("Warning: 'CLASS_NAMES_RAW' was not defined. Re-initialized.")
# --- End of model fix block ---

# --- Fix for NameError: name 'test_dataset' is not defined and FileNotFoundError ---
# This block ensures the dataset is re-initialized and present if the kernel state was lost.
if 'test_dataset' not in globals():
    print("Warning: 'test_dataset' was not defined. Re-initializing dataset components and data.")

    # ADDED: Check if Google Drive is mounted
    if False:
        print("\n=================================================================================")
        print("ERROR: Google Drive is not mounted.")
        print("=================================================================================\n")

    # Re-define RAW_TO_MODEL if not in globals (from cell yBpYcLb3YkyV)
    if 'RAW_TO_MODEL' not in globals():
        RAW_TO_MODEL = {0: 1, 1: 2} # Default values, assuming 2 classes plus background
        print("Warning: 'RAW_TO_MODEL' was not defined. Re-initialized.")

    # Re-define LOCAL_DATASET_PATH if not in globals (from cell qaavfk87VfKd)
    if 'LOCAL_DATASET_PATH' not in globals():
        LOCAL_DATASET_PATH = "LOCAL_DATASET_PATH" # Default path based on previous cells
        print("Warning: 'LOCAL_DATASET_PATH' was not defined. Re-initialized.")

    # Re-define TRAIN_DIR, VALID_DIR, TEST_DIR if not in globals (from cell Mb_3r118ZA1q)
    if 'TRAIN_DIR' not in globals():
        TRAIN_DIR = os.path.join(LOCAL_DATASET_PATH, "train")
        VALID_DIR = os.path.join(LOCAL_DATASET_PATH, "valid")
        TEST_DIR  = os.path.join(LOCAL_DATASET_PATH, "test")
        print("Warning: Dataset directories were not defined. Re-initialized.")

    # ADDED: Re-populate LOCAL_DATASET_PATH if the 'test/images' directory is missing
    if not os.path.exists(os.path.join(TEST_DIR, "images")):
        print("Warning: Local dataset content missing. Attempting to re-populate from Google Drive.")
        os.makedirs(LOCAL_DATASET_PATH, exist_ok=True)

        # Check for a zip file first
        zips = glob.glob(os.path.join(DATASET_DRIVE_PATH, "*.zip"))
        if zips:
            print(f"Found dataset zip file: {zips[0]}. Extracting...")
            with zipfile.ZipFile(zips[0], 'r') as z:
                z.extractall(LOCAL_DATASET_PATH)
            print("Dataset extracted.")
        else:
            # If no zip, assume direct directory structure (train, valid, test folders)
            expected_subdirs = ['train', 'valid', 'test']
            drive_contents = []
            try:
                if os.path.isdir(DATASET_DRIVE_PATH):
                    drive_contents = os.listdir(DATASET_DRIVE_PATH)
                else:
                    print(f"Warning: DATASET_DRIVE_PATH '{DATASET_DRIVE_PATH}' is not a directory or does not exist.")
            except Exception as e:
                print(f"Warning: Could not list DATASET_DRIVE_PATH '{DATASET_DRIVE_PATH}': {e}")

            all_subdirs_present_in_list = all(d in drive_contents for d in expected_subdirs)

            if all_subdirs_present_in_list:
                print(f"Verified expected dataset structure via os.listdir in '{DATASET_DRIVE_PATH}'. Proceeding with copy.")
                # Clear existing local dataset content if it exists to ensure a fresh copy
                if os.path.exists(LOCAL_DATASET_PATH) and len(os.listdir(LOCAL_DATASET_PATH)) > 0:
                    print(f"Clearing existing content in {LOCAL_DATASET_PATH} before copying.")
                    shutil.rmtree(LOCAL_DATASET_PATH)
                    os.makedirs(LOCAL_DATASET_PATH, exist_ok=True)
                try:
                    shutil.copytree(DATASET_DRIVE_PATH, LOCAL_DATASET_PATH, dirs_exist_ok=True)
                    print("Dataset copied successfully.")
                except Exception as e:
                    print(f"Error during dataset copy to {LOCAL_DATASET_PATH}: {e}")
                    print(f"Please ensure your Google Drive is mounted correctly and '{DATASET_DRIVE_PATH}' contains the 'train', 'valid', and 'test' subdirectories.")
                    raise FileNotFoundError("Dataset content not found in Google Drive after copy attempt.")
            else:
                missing_subdirs = set(expected_subdirs) - set(drive_contents)
                print(f"Error: DATASET_DRIVE_PATH '{DATASET_DRIVE_PATH}' does not contain all expected dataset subdirectories (missing: {list(missing_subdirs)}) or is inaccessible.")
                print("Please ensure your Google Drive is mounted correctly and the specified DATASET_DRIVE_PATH points to the correct location of your dataset.")
                raise FileNotFoundError("Dataset content not found in Google Drive.")
        print("Local dataset re-population attempt completed.")

    # Re-define YoloDetectionDataset class if not in globals (from cell yBpYcLb3YkyV)
    if 'YoloDetectionDataset' not in globals():
        class YoloDetectionDataset(Dataset):
            def __init__(self, split_dir, transforms=None):
                self.images_dir = os.path.join(split_dir, "images")
                self.labels_dir = os.path.join(split_dir, "labels")
                self.transforms = transforms
                self.image_files = sorted([
                    f for f in os.listdir(self.images_dir)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
                ])

            def __len__(self):
                return len(self.image_files)

            def __getitem__(self, idx):
                img_name = self.image_files[idx]
                img_path = os.path.join(self.images_dir, img_name)
                label_path = os.path.join(self.labels_dir, os.path.splitext(img_name)[0] + ".txt")

                img = Image.open(img_path).convert("RGB")
                img_w, img_h = img.size

                boxes, labels = [], []
                if os.path.exists(label_path):
                    with open(label_path) as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) != 5:
                                continue
                            cls, xc, yc, w, h = parts
                            cls = int(cls)
                            xc, yc, w, h = float(xc), float(yc), float(w), float(h)

                            x1 = (xc - w / 2) * img_w
                            y1 = (yc - h / 2) * img_h
                            x2 = (xc + w / 2) * img_w
                            y2 = (yc + h / 2) * img_h

                            if x2 <= x1 or y2 <= y1:
                                continue

                            boxes.append([x1, y1, x2, y2])
                            labels.append(RAW_TO_MODEL[cls]) # Ensure RAW_TO_MODEL is available

                boxes = torch.as_tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4), dtype=torch.float32)
                labels = torch.as_tensor(labels, dtype=torch.int64) if labels else torch.zeros((0,), dtype=torch.int64)

                target = {
                    "boxes": boxes,
                    "labels": labels,
                    "image_id": torch.tensor([idx]),
                    "area": (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]) if len(boxes) else torch.zeros((0,)),
                    "iscrowd": torch.zeros((len(boxes),), dtype=torch.int64),
                }

                img = F.to_tensor(img)
                if self.transforms:
                    img, target = self.transforms(img, target)
                return img, target
        print("Warning: YoloDetectionDataset class was not defined. Re-defined.")

    test_dataset = YoloDetectionDataset(TEST_DIR)
    print("Warning: 'test_dataset' was instantiated.")
# --- End of test_dataset fix block ---

model.eval()

with torch.no_grad():
    for idx in range(min(NUM_IMAGES, len(test_dataset))):

        # Get test image
        image, target = test_dataset[idx]

        # Move image to GPU/CPU
        image_device = image.to(device)

        # Prediction
        prediction = model([image_device])[0]

        # Move everything back to CPU
        image_cpu = image.cpu()

        boxes = prediction["boxes"].cpu()
        labels = prediction["labels"].cpu()
        scores = prediction["scores"].cpu()

        # Convert tensor to image
        image_pil = to_pil_image(image_cpu)

        # Display
        plt.figure(figsize=(12, 8))
        plt.imshow(image_pil)
        ax = plt.gca()

        # Draw predictions
        for box, label, score in zip(boxes, labels, scores):

            if score < SCORE_THRESH:
                continue

            raw_label = MODEL_TO_RAW.get(int(label), int(label))
            display_label = CLASS_NAMES_RAW.get(raw_label, str(raw_label))

            x1, y1, x2, y2 = box.tolist()

            rect = plt.Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                fill=False,
                linewidth=2
            )

            ax.add_patch(rect)

            ax.text(
                x1,
                y1,
                f"{display_label}: {score.item():.2f}",
                fontsize=10,
                bbox=dict(alpha=0.7)
            )

        plt.axis("off")
        plt.tight_layout(pad=0)

        # Save to Google Drive
        save_path = os.path.join(
            SAVE_DIR,
            f"test_prediction_{idx+1:02d}.png"
        )

        plt.savefig(
            save_path,
            dpi=150,
            bbox_inches="tight",
            pad_inches=0
        )

        plt.close()

        print(f"Saved: {save_path}")

print("\nDone!")
print(f"Saved {NUM_IMAGES} test prediction images to:")
print(SAVE_DIR)

In [ ]:
# ============================================================
# FASTER R-CNN - COUNT PEOPLE ON LAND AND WATER
# ============================================================

import os
import torch
import torchvision
import matplotlib.pyplot as plt
from PIL import Image
from torchvision.transforms import functional as F
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# ------------------------------------------------------------
# 1. MOUNTED DRIVE PATHS
# ------------------------------------------------------------

TEST_IMAGE_DIR = os.path.join(LOCAL_DATASET_PATH, "test", "images")
MODEL_PATH = os.path.join(LOCAL_DATASET_PATH, "fasterrcnn_beach_human.pth")

# ------------------------------------------------------------
# 2. DEVICE
# ------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ------------------------------------------------------------
# 3. CREATE MODEL
# ------------------------------------------------------------

# 3 classes:
# 0 = background
# 1 = person on land
# 2 = person on water

NUM_CLASSES = 3

model = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(
    weights=None
)

in_features = model.roi_heads.box_predictor.cls_score.in_features

model.roi_heads.box_predictor = FastRCNNPredictor(
    in_features,
    NUM_CLASSES
)

# ------------------------------------------------------------
# 4. LOAD TRAINED WEIGHTS
# ------------------------------------------------------------

print("Loading model...")

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device
)

model.load_state_dict(checkpoint)

model.to(device)
model.eval()

print("✅ Model loaded successfully!")

# ------------------------------------------------------------
# 5. SETTINGS
# ------------------------------------------------------------

CONFIDENCE_THRESHOLD = 0.60

# Number of test images to display
NUM_IMAGES = 6

# ------------------------------------------------------------
# 6. GET TEST IMAGES
# ------------------------------------------------------------

image_files = sorted([
    os.path.join(TEST_IMAGE_DIR, f)
    for f in os.listdir(TEST_IMAGE_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

print("Total test images:", len(image_files))

# ------------------------------------------------------------
# 7. PREDICT AND DISPLAY
# ------------------------------------------------------------

for image_path in image_files[:NUM_IMAGES]:

    image = Image.open(image_path).convert("RGB")

    image_tensor = F.to_tensor(image).to(device)

    # Prediction
    with torch.no_grad():
        output = model([image_tensor])[0]

    boxes = output["boxes"].cpu()
    labels = output["labels"].cpu()
    scores = output["scores"].cpu()

    # --------------------------------------------------------
    # FILTER LOW CONFIDENCE DETECTIONS
    # --------------------------------------------------------

    keep = scores >= CONFIDENCE_THRESHOLD

    boxes = boxes[keep]
    labels = labels[keep]
    scores = scores[keep]

    # --------------------------------------------------------
    # COUNT EACH CLASS
    # --------------------------------------------------------

    land_count = int((labels == 1).sum())
    water_count = int((labels == 2).sum())

    total_count = land_count + water_count

    # --------------------------------------------------------
    # DISPLAY
    # --------------------------------------------------------

    fig, ax = plt.subplots(figsize=(12, 8))

    ax.imshow(image)

    # Draw bounding boxes
    for box, label, score in zip(boxes, labels, scores):

        x1, y1, x2, y2 = box.tolist()

        if label.item() == 1:
            class_name = "Person on Land"
        elif label.item() == 2:
            class_name = "Person on Water"
        else:
            continue

        rect = plt.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            fill=False,
            linewidth=2
        )

        ax.add_patch(rect)

        ax.text(
            x1,
            max(y1 - 5, 0),
            f"{class_name} ({score:.2f})",
            fontsize=9,
            backgroundcolor="white"
        )

    # --------------------------------------------------------
    # COUNT TEXT
    # --------------------------------------------------------

    count_text = (
        f"PERSON ON LAND  : {land_count}\n"
        f"PERSON ON WATER : {water_count}\n"
        f"TOTAL PEOPLE    : {total_count}"
    )

    ax.text(
        0.02,
        0.03,
        count_text,
        transform=ax.transAxes,
        fontsize=14,
        fontweight="bold",
        verticalalignment="bottom",
        backgroundcolor="white"
    )

    ax.set_title(
        f"Predictions - {os.path.basename(image_path)}"
    )

    ax.axis("off")

    plt.show()

    # --------------------------------------------------------
    # PRINT COUNTS
    # --------------------------------------------------------

    print(
        f"{os.path.basename(image_path)} | "
        f"Land = {land_count} | "
        f"Water = {water_count} | "
        f"Total = {total_count}"
    )

In [ ]:
# ============================================================
# FASTER R-CNN - TEST IMAGES WITH COUNTS
# ============================================================

import os
import torch
import torchvision
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from torchvision.transforms import functional as F
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

TEST_IMAGE_DIR = os.path.join(LOCAL_DATASET_PATH, "test", "images")
MODEL_PATH = os.path.join(LOCAL_DATASET_PATH, "fasterrcnn_beach_human.pth")

# ------------------------------------------------------------
# 2. DEVICE
# ------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

# ------------------------------------------------------------
# 3. CREATE MODEL
# ------------------------------------------------------------

NUM_CLASSES = 3

model = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(
    weights=None
)

in_features = model.roi_heads.box_predictor.cls_score.in_features

model.roi_heads.box_predictor = FastRCNNPredictor(
    in_features,
    NUM_CLASSES
)

# ------------------------------------------------------------
# 4. LOAD TRAINED MODEL
# ------------------------------------------------------------

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device
)

model.load_state_dict(checkpoint)

model.to(device)
model.eval()

print("✅ Model loaded successfully!")

# ------------------------------------------------------------
# 5. SETTINGS
# ------------------------------------------------------------

CONFIDENCE_THRESHOLD = 0.60
NUM_IMAGES = 6

# Same mapping used in your notebook:
# Model label 1 -> Raw class 0 -> Land
# Model label 2 -> Raw class 1 -> Water

MODEL_TO_RAW = {
    1: 0,
    2: 1
}

# ------------------------------------------------------------
# 6. GET TEST IMAGES
# ------------------------------------------------------------

image_files = sorted([
    os.path.join(TEST_IMAGE_DIR, f)
    for f in os.listdir(TEST_IMAGE_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))
])

print("Total test images:", len(image_files))

# ------------------------------------------------------------
# 7. TEST AND DISPLAY
# ------------------------------------------------------------

for image_path in image_files[:NUM_IMAGES]:

    # Load image
    image = Image.open(image_path).convert("RGB")

    # Convert to tensor
    img_tensor = F.to_tensor(image).to(device)

    # Prediction
    with torch.no_grad():
        pred = model([img_tensor])[0]

    # --------------------------------------------------------
    # FILTER PREDICTIONS
    # --------------------------------------------------------

    boxes = pred["boxes"].cpu()
    labels = pred["labels"].cpu()
    scores = pred["scores"].cpu()

    keep = scores >= CONFIDENCE_THRESHOLD

    boxes = boxes[keep]
    labels = labels[keep]
    scores = scores[keep]

    # --------------------------------------------------------
    # COUNT PEOPLE
    # --------------------------------------------------------

    # Model label 1 = raw class 0 = land
    # Model label 2 = raw class 1 = water

    land_count = int((labels == 1).sum())
    water_count = int((labels == 2).sum())

    total_count = land_count + water_count

    # --------------------------------------------------------
    # DISPLAY
    # --------------------------------------------------------

    fig, ax = plt.subplots(figsize=(12, 8))

    ax.imshow(image)

    # --------------------------------------------------------
    # DRAW BOXES + ORIGINAL NOTEBOOK-STYLE TAGS
    # --------------------------------------------------------

    for box, label, score in zip(boxes, labels, scores):

        # Convert model label back to original dataset label
        raw_label = MODEL_TO_RAW.get(
            int(label),
            int(label)
        )

        x1, y1, x2, y2 = box.numpy()

        # Green bounding box
        rect = patches.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            linewidth=2,
            edgecolor="lime",
            facecolor="none"
        )

        ax.add_patch(rect)

        # EXACT STYLE OF YOUR PREVIOUS NOTEBOOK:
        # 0:confidence = land
        # 1:confidence = water
        ax.text(
            x1,
            max(y1 - 5, 5),
            f"{raw_label}:{score:.2f}",
            color="lime",
            fontsize=10,
            weight="bold",
            verticalalignment="bottom"
        )

    # --------------------------------------------------------
    # COUNT PANEL AT TOP
    # --------------------------------------------------------

    count_text = (
        f"Land (0): {land_count}    "
        f"Water (1): {water_count}    "
        f"Total: {total_count}"
    )

    ax.text(
        0.02,
        0.98,
        count_text,
        transform=ax.transAxes,
        fontsize=11,
        weight="bold",
        color="white",
        verticalalignment="top",
        bbox=dict(
            boxstyle="round,pad=0.4",
            facecolor="black",
            alpha=0.65,
            edgecolor="none"
        )
    )

    # --------------------------------------------------------
    # CLEAN DISPLAY
    # --------------------------------------------------------

    ax.axis("off")

    plt.tight_layout()

    plt.show()

    # --------------------------------------------------------
    # PRINT COUNTS
    # --------------------------------------------------------

    print(
        f"{os.path.basename(image_path)} | "
        f"Land (0) = {land_count} | "
        f"Water (1) = {water_count} | "
        f"Total = {total_count}"
    )

In [ ]:
# ============================================================
# LIVE IMAGE INPUT + PEOPLE COUNTING
# 0 = LAND
# 1 = WATER
# ============================================================

import os
import torch
import torchvision
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from PIL import Image
from torchvision.transforms import functional as F
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor


# ------------------------------------------------------------
# 1. MODEL PATH
# ------------------------------------------------------------

MODEL_PATH = os.path.join(LOCAL_DATASET_PATH, "fasterrcnn_beach_human.pth")

# ------------------------------------------------------------
# 2. DEVICE
# ------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

# ------------------------------------------------------------
# 3. CREATE MODEL
# ------------------------------------------------------------

NUM_CLASSES = 3

model = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(
    weights=None
)

in_features = model.roi_heads.box_predictor.cls_score.in_features

model.roi_heads.box_predictor = FastRCNNPredictor(
    in_features,
    NUM_CLASSES
)

# ------------------------------------------------------------
# 4. LOAD TRAINED MODEL
# ------------------------------------------------------------

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device
)

model.load_state_dict(checkpoint)

model.to(device)
model.eval()

print("✅ Model loaded successfully!")

# ------------------------------------------------------------
# 5. SETTINGS
# ------------------------------------------------------------

CONFIDENCE_THRESHOLD = 0.60

# Same mapping as your notebook:
# Model label 1 -> raw class 0 -> Land
# Model label 2 -> raw class 1 -> Water

MODEL_TO_RAW = {
    1: 0,
    2: 1
}

# ------------------------------------------------------------
# 6. UPLOAD IMAGE
# ------------------------------------------------------------

print("\n📷 Enter the path of an image available in Kaggle:")
image_name = input("Image path: ").strip()

if not os.path.exists(image_name):
    print("❌ Image not found:", image_name)

else:

    image = Image.open(image_name).convert("RGB")

    # --------------------------------------------------------
    # 7. CONVERT IMAGE TO TENSOR
    # --------------------------------------------------------

    image_tensor = F.to_tensor(image).to(device)

    # --------------------------------------------------------
    # 8. RUN FASTER R-CNN
    # --------------------------------------------------------

    with torch.no_grad():

        prediction = model([image_tensor])[0]

    boxes = prediction["boxes"].cpu()
    labels = prediction["labels"].cpu()
    scores = prediction["scores"].cpu()

    # --------------------------------------------------------
    # 9. CONFIDENCE FILTER
    # --------------------------------------------------------

    keep = scores >= CONFIDENCE_THRESHOLD

    boxes = boxes[keep]
    labels = labels[keep]
    scores = scores[keep]

    # --------------------------------------------------------
    # 10. COUNT PEOPLE
    # --------------------------------------------------------

    # Model label 1 = original class 0 = Land
    land_count = int((labels == 1).sum())

    # Model label 2 = original class 1 = Water
    water_count = int((labels == 2).sum())

    total_count = land_count + water_count

    # --------------------------------------------------------
    # 11. DISPLAY IMAGE
    # --------------------------------------------------------

    fig, ax = plt.subplots(figsize=(12, 8))

    ax.imshow(image)

    # --------------------------------------------------------
    # 12. DRAW GREEN BOXES + 0/1 TAGS
    # --------------------------------------------------------

    for box, label, score in zip(boxes, labels, scores):

        # Convert model label back to original label
        raw_label = MODEL_TO_RAW.get(
            int(label),
            int(label)
        )

        x1, y1, x2, y2 = box.numpy()

        # Green bounding box
        rect = patches.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            linewidth=2,
            edgecolor="lime",
            facecolor="none"
        )

        ax.add_patch(rect)

        # ----------------------------------------------------
        # TAG
        # ----------------------------------------------------
        # 0 = Land
        # 1 = Water

        ax.text(
            x1,
            max(y1 - 5, 5),
            f"{raw_label}:{score:.2f}",
            color="lime",
            fontsize=9,
            weight="bold",
            verticalalignment="bottom"
        )

    # --------------------------------------------------------
    # 13. COUNT AT TOP OF IMAGE
    # --------------------------------------------------------

    count_text = (
        f"Land (0): {land_count}    "
        f"Water (1): {water_count}    "
        f"Total: {total_count}"
    )

    ax.text(
        0.02,
        0.98,
        count_text,
        transform=ax.transAxes,
        fontsize=11,
        weight="bold",
        color="white",
        verticalalignment="top",
        bbox=dict(
            boxstyle="round,pad=0.35",
            facecolor="black",
            alpha=0.65,
            edgecolor="none"
        )
    )

    # --------------------------------------------------------
    # 14. CLEAN OUTPUT
    # --------------------------------------------------------

    ax.axis("off")

    plt.tight_layout()

    plt.show()

    # --------------------------------------------------------
    # 15. PRINT FINAL COUNT
    # --------------------------------------------------------

    print("\n========== LIVE COUNT ==========")
    print(f"People on Land  (0): {land_count}")
    print(f"People on Water (1): {water_count}")
    print(f"Total People       : {total_count}")
    print("================================")

In [ ]:
# ============================================================
# TEST NEXT 10 IMAGES
# 0 = LAND
# 1 = WATER
# ============================================================

import os
import torch
import torchvision
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from PIL import Image
from torchvision.transforms import functional as F
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

TEST_IMAGE_DIR = os.path.join(LOCAL_DATASET_PATH, "test", "images")
MODEL_PATH = os.path.join(LOCAL_DATASET_PATH, "fasterrcnn_beach_human.pth")

# ------------------------------------------------------------
# 2. DEVICE
# ------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

# ------------------------------------------------------------
# 3. CREATE MODEL
# ------------------------------------------------------------

NUM_CLASSES = 3

model = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(
    weights=None
)

in_features = model.roi_heads.box_predictor.cls_score.in_features

model.roi_heads.box_predictor = FastRCNNPredictor(
    in_features,
    NUM_CLASSES
)

# ------------------------------------------------------------
# 4. LOAD TRAINED MODEL
# ------------------------------------------------------------

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device
)

model.load_state_dict(checkpoint)

model.to(device)
model.eval()

print("✅ Model loaded successfully!")

# ------------------------------------------------------------
# 5. SETTINGS
# ------------------------------------------------------------

CONFIDENCE_THRESHOLD = 0.60

# Start after the first 6 images
START_INDEX = 6

# Test 10 images
NUM_IMAGES = 10

# Original notebook mapping
MODEL_TO_RAW = {
    1: 0,   # Model 1 -> class 0 -> Land
    2: 1    # Model 2 -> class 1 -> Water
}

# ------------------------------------------------------------
# 6. GET TEST IMAGES
# ------------------------------------------------------------

image_files = sorted([
    os.path.join(TEST_IMAGE_DIR, f)
    for f in os.listdir(TEST_IMAGE_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))
])

print("Total test images:", len(image_files))

# Select images 7 to 16
selected_images = image_files[
    START_INDEX : START_INDEX + NUM_IMAGES
]

print(
    f"Testing images {START_INDEX + 1} "
    f"to {START_INDEX + len(selected_images)}"
)

# ------------------------------------------------------------
# 7. PREDICT EACH IMAGE
# ------------------------------------------------------------

for image_path in selected_images:

    # --------------------------------------------------------
    # LOAD IMAGE
    # --------------------------------------------------------

    image = Image.open(image_path).convert("RGB")

    image_tensor = F.to_tensor(image).to(device)

    # --------------------------------------------------------
    # MODEL PREDICTION
    # --------------------------------------------------------

    with torch.no_grad():

        prediction = model([image_tensor])[0]

    boxes = prediction["boxes"].cpu()
    labels = prediction["labels"].cpu()
    scores = prediction["scores"].cpu()

    # --------------------------------------------------------
    # CONFIDENCE FILTER
    # --------------------------------------------------------

    keep = scores >= CONFIDENCE_THRESHOLD

    boxes = boxes[keep]
    labels = labels[keep]
    scores = scores[keep]

    # --------------------------------------------------------
    # COUNT LAND AND WATER
    # --------------------------------------------------------

    land_count = int((labels == 1).sum())
    water_count = int((labels == 2).sum())

    total_count = land_count + water_count

    # --------------------------------------------------------
    # DISPLAY IMAGE
    # --------------------------------------------------------

    fig, ax = plt.subplots(figsize=(12, 8))

    ax.imshow(image)

    # --------------------------------------------------------
    # GREEN BOXES + 0/1 TAGS
    # --------------------------------------------------------

    for box, label, score in zip(
        boxes,
        labels,
        scores
    ):

        # Convert model label to original dataset label
        raw_label = MODEL_TO_RAW.get(
            int(label),
            int(label)
        )

        x1, y1, x2, y2 = box.numpy()

        # Green bounding box
        rect = patches.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            linewidth=2,
            edgecolor="lime",
            facecolor="none"
        )

        ax.add_patch(rect)

        # Same tag format as your original notebook
        ax.text(
            x1,
            max(y1 - 5, 5),
            f"{raw_label}:{score:.2f}",
            color="lime",
            fontsize=9,
            weight="bold",
            verticalalignment="bottom"
        )

    # --------------------------------------------------------
    # COUNT AT TOP
    # --------------------------------------------------------

    count_text = (
        f"Land (0): {land_count}    "
        f"Water (1): {water_count}    "
        f"Total: {total_count}"
    )

    ax.text(
        0.02,
        0.98,
        count_text,
        transform=ax.transAxes,
        fontsize=11,
        weight="bold",
        color="white",
        verticalalignment="top",
        bbox=dict(
            boxstyle="round,pad=0.35",
            facecolor="black",
            alpha=0.65,
            edgecolor="none"
        )
    )

    # --------------------------------------------------------
    # CLEAN DISPLAY
    # --------------------------------------------------------

    ax.axis("off")

    plt.tight_layout()

    plt.show()

    # --------------------------------------------------------
    # PRINT RESULT
    # --------------------------------------------------------

    print(
        f"{os.path.basename(image_path)} | "
        f"Land (0) = {land_count} | "
        f"Water (1) = {water_count} | "
        f"Total = {total_count}"
    )

In [ ]:
# ============================================================
# FIX: MOUNT DRIVE + CHECK MODEL FILE
# ============================================================

import os


MODEL_PATH = os.path.join(LOCAL_DATASET_PATH, "fasterrcnn_beach_human.pth")
TEST_IMAGE_DIR = os.path.join(LOCAL_DATASET_PATH, "test", "images")

print("Model exists:", os.path.exists(MODEL_PATH))
print("Test folder exists:", os.path.exists(TEST_IMAGE_DIR))

if os.path.exists(MODEL_PATH):
    print("✅ Model found:")
    print(MODEL_PATH)
else:
    print("❌ Model NOT found.")
    print("\nFiles in dataset folder:")
    print(os.listdir("LOCAL_DATASET_PATH"))

In [ ]:
# ============================================================
# AUTOMATICALLY REMOVE NON-HUMAN BOUNDING BOXES
#
# RAW CLASSES:
# 0 = HUMAN ON LAND
# 1 = HUMAN ON WATER
#
# This code:
# 1. Mounts Google Drive
# 2. Installs YOLO
# 3. Downloads a pretrained person detector
# 4. Checks every existing annotation
# 5. Keeps boxes that match REAL PERSON detections
# 6. Removes boxes around poles / objects
# 7. Keeps the original 0/1 class
# 8. Creates a backup of the original labels
# ============================================================

# ------------------------------------------------------------
# 1. MOUNT GOOGLE DRIVE
# ------------------------------------------------------------



# ------------------------------------------------------------
# 2. INSTALL ULTRALYTICS
# ------------------------------------------------------------

!pip -q install ultralytics


# ------------------------------------------------------------
# 3. IMPORTS
# ------------------------------------------------------------

import os
import shutil
import numpy as np
from PIL import Image
from ultralytics import YOLO


# ------------------------------------------------------------
# 4. DATASET PATH
# ------------------------------------------------------------

DATASET_PATH = "LOCAL_DATASET_PATH"

SPLITS = {
    "train": (
        os.path.join(DATASET_PATH, "train", "images"),
        os.path.join(DATASET_PATH, "train", "labels")
    ),

    "valid": (
        os.path.join(DATASET_PATH, "valid", "images"),
        os.path.join(DATASET_PATH, "valid", "labels")
    ),

    "test": (
        os.path.join(DATASET_PATH, "test", "images"),
        os.path.join(DATASET_PATH, "test", "labels")
    )
}


# ------------------------------------------------------------
# 5. BACKUP ORIGINAL LABELS
# ------------------------------------------------------------

BACKUP_PATH = os.path.join(
    DATASET_PATH,
    "labels_backup_before_human_cleaning"
)

os.makedirs(BACKUP_PATH, exist_ok=True)

print("Creating backup of original labels...")

for split, (image_dir, label_dir) in SPLITS.items():

    if not os.path.exists(label_dir):
        print(f"⚠️ Label folder not found: {label_dir}")
        continue

    backup_split = os.path.join(
        BACKUP_PATH,
        split
    )

    os.makedirs(
        backup_split,
        exist_ok=True
    )

    for filename in os.listdir(label_dir):

        if filename.endswith(".txt"):

            src = os.path.join(
                label_dir,
                filename
            )

            dst = os.path.join(
                backup_split,
                filename
            )

            shutil.copy2(src, dst)

print("✅ Original labels backed up to:")
print(BACKUP_PATH)


# ------------------------------------------------------------
# 6. LOAD PRETRAINED PERSON DETECTOR
# ------------------------------------------------------------
#
# YOLO class 0 = person in COCO
#
# We use it ONLY to determine whether an existing
# annotation actually corresponds to a human.
# Your original class 0/1 labels are NOT changed.
# ------------------------------------------------------------

print("\nLoading pretrained person detector...")

person_model = YOLO("yolov8m.pt")

print("✅ Person detector loaded")


# ------------------------------------------------------------
# 7. SETTINGS
# ------------------------------------------------------------

PERSON_CONFIDENCE = 0.20

# Minimum overlap between your existing annotation
# and the pretrained person's bounding box.
#
# Lower = keeps more boxes
# Higher = more aggressive cleaning

IOU_THRESHOLD = 0.20

# Also keep a box if its center lies inside
# a detected person box.
USE_CENTER_TEST = True


# ------------------------------------------------------------
# 8. IOU FUNCTION
# ------------------------------------------------------------

def calculate_iou(box1, box2):

    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])

    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection_width = max(
        0,
        x2 - x1
    )

    intersection_height = max(
        0,
        y2 - y1
    )

    intersection = (
        intersection_width *
        intersection_height
    )

    area1 = max(
        0,
        box1[2] - box1[0]
    ) * max(
        0,
        box1[3] - box1[1]
    )

    area2 = max(
        0,
        box2[2] - box2[0]
    ) * max(
        0,
        box2[3] - box2[1]
    )

    union = area1 + area2 - intersection

    if union <= 0:
        return 0.0

    return intersection / union


# ------------------------------------------------------------
# 9. CENTER-IN-BOX TEST
# ------------------------------------------------------------

def center_inside(box, person_box):

    cx = (box[0] + box[2]) / 2
    cy = (box[1] + box[3]) / 2

    return (
        person_box[0] <= cx <= person_box[2]
        and
        person_box[1] <= cy <= person_box[3]
    )


# ------------------------------------------------------------
# 10. PROCESS ONE IMAGE
# ------------------------------------------------------------

def clean_label_file(image_path, label_path):

    image = Image.open(image_path).convert("RGB")

    width, height = image.size

    # --------------------------------------------------------
    # READ EXISTING YOLO LABELS
    # --------------------------------------------------------

    if not os.path.exists(label_path):
        return 0, 0

    with open(label_path, "r") as f:
        lines = [
            line.strip()
            for line in f
            if line.strip()
        ]

    if len(lines) == 0:
        return 0, 0

    original_count = 0
    kept_count = 0

    annotation_data = []

    for line in lines:

        parts = line.split()

        if len(parts) != 5:
            continue

        try:
            class_id = int(parts[0])

            xc = float(parts[1])
            yc = float(parts[2])
            bw = float(parts[3])
            bh = float(parts[4])

        except:
            continue

        # Only allow your two human classes
        if class_id not in [0, 1]:
            continue

        # Convert YOLO normalized coordinates -> pixels

        x_center = xc * width
        y_center = yc * height

        box_width = bw * width
        box_height = bh * height

        x1 = x_center - box_width / 2
        y1 = y_center - box_height / 2

        x2 = x_center + box_width / 2
        y2 = y_center + box_height / 2

        annotation_data.append({
            "class_id": class_id,
            "yolo_line": line,
            "box": [
                x1,
                y1,
                x2,
                y2
            ]
        })

        original_count += 1


    # --------------------------------------------------------
    # RUN PRETRAINED PERSON DETECTOR
    # --------------------------------------------------------

    results = person_model.predict(
        source=image,
        conf=PERSON_CONFIDENCE,
        classes=[0],       # COCO PERSON ONLY
        verbose=False
    )

    result = results[0]

    detected_person_boxes = []

    if result.boxes is not None:

        for box, conf in zip(
            result.boxes.xyxy.cpu().numpy(),
            result.boxes.conf.cpu().numpy()
        ):

            if conf >= PERSON_CONFIDENCE:

                detected_person_boxes.append(
                    box.tolist()
                )


    # --------------------------------------------------------
    # FILTER EXISTING ANNOTATIONS
    # --------------------------------------------------------

    kept_lines = []

    for annotation in annotation_data:

        annotation_box = annotation["box"]

        is_human = False

        for person_box in detected_person_boxes:

            iou = calculate_iou(
                annotation_box,
                person_box
            )

            # Method 1: IoU
            if iou >= IOU_THRESHOLD:

                is_human = True
                break

            # Method 2: center inside detected person
            if USE_CENTER_TEST:

                if center_inside(
                    annotation_box,
                    person_box
                ):

                    is_human = True
                    break


        # ----------------------------------------------------
        # KEEP ONLY HUMAN BOXES
        # ----------------------------------------------------

        if is_human:

            kept_lines.append(
                annotation["yolo_line"]
            )

            kept_count += 1


    # --------------------------------------------------------
    # WRITE CLEAN LABEL FILE
    # --------------------------------------------------------

    with open(label_path, "w") as f:

        if kept_lines:

            f.write(
                "\n".join(kept_lines)
                + "\n"
            )

        else:
            # Empty label file = no humans
            f.write("")


    return original_count, kept_count


# ------------------------------------------------------------
# 11. PROCESS ENTIRE DATASET
# ------------------------------------------------------------

total_original = 0
total_kept = 0
total_removed = 0

for split, (image_dir, label_dir) in SPLITS.items():

    print("\n" + "=" * 60)
    print(f"PROCESSING {split.upper()}")
    print("=" * 60)

    if not os.path.exists(image_dir):
        print("⚠️ Image folder not found:")
        print(image_dir)
        continue

    os.makedirs(
        label_dir,
        exist_ok=True
    )

    image_files = sorted([
        f for f in os.listdir(image_dir)
        if f.lower().endswith(
            (".jpg", ".jpeg", ".png", ".bmp")
        )
    ])

    split_original = 0
    split_kept = 0

    for index, filename in enumerate(image_files):

        image_path = os.path.join(
            image_dir,
            filename
        )

        label_filename = (
            os.path.splitext(filename)[0]
            + ".txt"
        )

        label_path = os.path.join(
            label_dir,
            label_filename
        )

        original, kept = clean_label_file(
            image_path,
            label_path
        )

        split_original += original
        split_kept += kept

        if original > kept:

            removed = original - kept

            print(
                f"[{index+1}/{len(image_files)}] "
                f"{filename}: "
                f"{original} → {kept} "
                f"(removed {removed})"
            )

    split_removed = (
        split_original -
        split_kept
    )

    total_original += split_original
    total_kept += split_kept
    total_removed += split_removed

    print(f"\n{split.upper()} SUMMARY")
    print("Original boxes :", split_original)
    print("Human boxes    :", split_kept)
    print("Removed boxes  :", split_removed)


# ------------------------------------------------------------
# 12. FINAL SUMMARY
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("              CLEANING COMPLETE")
print("=" * 60)

print("Original boxes :", total_original)
print("Human boxes    :", total_kept)
print("Removed boxes  :", total_removed)

print("\n✅ Original labels are safely backed up at:")
print(BACKUP_PATH)

print("\nYour cleaned dataset is still at:")
print(DATASET_PATH)

print("\nNext step:")
print("Retrain Faster R-CNN using the cleaned labels.")

In [ ]:
# ============================================================
# RETRAIN FASTER R-CNN ON CLEAN HUMAN-ONLY LABELS
#
# RAW CLASS:
# 0 = HUMAN ON LAND
# 1 = HUMAN ON WATER
# ============================================================

import os
import torch
import torchvision

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as F
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

DATASET_PATH = "LOCAL_DATASET_PATH"

TRAIN_IMG_DIR = os.path.join(DATASET_PATH, "train", "images")
TRAIN_LABEL_DIR = os.path.join(DATASET_PATH, "train", "labels")

VAL_IMG_DIR = os.path.join(DATASET_PATH, "valid", "images")
VAL_LABEL_DIR = os.path.join(DATASET_PATH, "valid", "labels")

NEW_MODEL_PATH = os.path.join(
    DATASET_PATH,
    "fasterrcnn_beach_human_clean.pth"
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

# ------------------------------------------------------------
# CLASS MAPPING
# ------------------------------------------------------------

# YOLO/raw:
# 0 = land
# 1 = water
#
# Faster R-CNN:
# 0 = background
# 1 = land
# 2 = water

RAW_TO_MODEL = {
    0: 1,
    1: 2
}

NUM_CLASSES = 3


# ------------------------------------------------------------
# DATASET
# ------------------------------------------------------------

class BeachHumanDataset(Dataset):

    def __init__(self, image_dir, label_dir):
        self.image_dir = image_dir
        self.label_dir = label_dir

        self.images = sorted([
            f for f in os.listdir(image_dir)
            if f.lower().endswith(
                (".jpg", ".jpeg", ".png", ".bmp")
            )
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        filename = self.images[idx]

        image_path = os.path.join(
            self.image_dir,
            filename
        )

        image = Image.open(image_path).convert("RGB")

        width, height = image.size

        label_filename = os.path.splitext(filename)[0] + ".txt"

        label_path = os.path.join(
            self.label_dir,
            label_filename
        )

        boxes = []
        labels = []

        # ----------------------------------------------------
        # EMPTY LABEL FILE = NO HUMANS
        # ----------------------------------------------------

        if os.path.exists(label_path):

            with open(label_path, "r") as f:

                for line in f:

                    parts = line.strip().split()

                    if len(parts) != 5:
                        continue

                    raw_class = int(parts[0])

                    # Ignore unknown classes
                    if raw_class not in RAW_TO_MODEL:
                        continue

                    xc = float(parts[1]) * width
                    yc = float(parts[2]) * height
                    bw = float(parts[3]) * width
                    bh = float(parts[4]) * height

                    x1 = xc - bw / 2
                    y1 = yc - bh / 2
                    x2 = xc + bw / 2
                    y2 = yc + bh / 2

                    # Prevent invalid boxes
                    x1 = max(0, min(x1, width))
                    y1 = max(0, min(y1, height))
                    x2 = max(0, min(x2, width))
                    y2 = max(0, min(y2, height))

                    if x2 <= x1 or y2 <= y1:
                        continue

                    boxes.append([x1, y1, x2, y2])
                    labels.append(RAW_TO_MODEL[raw_class])

        # ----------------------------------------------------
        # TENSORS
        # ----------------------------------------------------

        boxes = torch.tensor(
            boxes,
            dtype=torch.float32
        ).reshape(-1, 4)

        labels = torch.tensor(
            labels,
            dtype=torch.int64
        )

        image_tensor = F.to_tensor(image)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
        }

        return image_tensor, target


# ------------------------------------------------------------
# COLLATE FUNCTION
# ------------------------------------------------------------

def collate_fn(batch):
    return tuple(zip(*batch))


# ------------------------------------------------------------
# DATASETS
# ------------------------------------------------------------

train_dataset = BeachHumanDataset(
    TRAIN_IMG_DIR,
    TRAIN_LABEL_DIR
)

val_dataset = BeachHumanDataset(
    VAL_IMG_DIR,
    VAL_LABEL_DIR
)

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2
)

print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))


# ------------------------------------------------------------
# MODEL
# ------------------------------------------------------------

model = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(
    weights="DEFAULT"
)

in_features = (
    model.roi_heads
    .box_predictor
    .cls_score
    .in_features
)

model.roi_heads.box_predictor = FastRCNNPredictor(
    in_features,
    NUM_CLASSES
)

model.to(device)


# ------------------------------------------------------------
# OPTIMIZER
# ------------------------------------------------------------

params = [
    p for p in model.parameters()
    if p.requires_grad
]

optimizer = torch.optim.SGD(
    params,
    lr=0.005,
    momentum=0.9,
    weight_decay=0.0005
)

lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=5,
    gamma=0.1
)


# ------------------------------------------------------------
# TRAIN
# ------------------------------------------------------------

NUM_EPOCHS = 15000

for epoch in range(NUM_EPOCHS):

    model.train()

    total_loss = 0.0

    for images, targets in train_loader:

        images = [
            img.to(device)
            for img in images
        ]

        targets = [
            {
                k: v.to(device)
                for k, v in t.items()
            }
            for t in targets
        ]

        loss_dict = model(
            images,
            targets
        )

        losses = sum(
            loss for loss in loss_dict.values()
        )

        optimizer.zero_grad()

        losses.backward()

        optimizer.step()

        total_loss += losses.item()

    lr_scheduler.step()

    avg_loss = total_loss / max(
        len(train_loader),
        1
    )

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] "
        f"Loss: {avg_loss:.4f}"
    )


# ------------------------------------------------------------
# SAVE CLEAN MODEL
# ------------------------------------------------------------

torch.save(
    model.state_dict(),
    NEW_MODEL_PATH
)

print("\n======================================")
print("✅ CLEAN MODEL TRAINING COMPLETE")
print("======================================")
print("Saved model:")
print(NEW_MODEL_PATH)

In [ ]:
# ============================================================
# TEST EXACTLY 10 AVAILABLE TEST IMAGES
# AUTOMATICALLY REPLACES MISSING REQUESTED IMAGES
#
# 0 = LAND
# 1 = WATER
# ============================================================

import os
import shutil
import numpy as np
import pandas as pd
import torch
import torchvision
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from PIL import Image
from torchvision.transforms import functional as F
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_curve,
    average_precision_score
)

# ============================================================
# 1. PATHS
# ============================================================

DATASET_PATH = "LOCAL_DATASET_PATH"

TEST_IMAGE_DIR = os.path.join(
    DATASET_PATH,
    "test",
    "images"
)

TEST_LABEL_DIR = os.path.join(
    DATASET_PATH,
    "test",
    "labels"
)

CLEAN_MODEL = os.path.join(
    DATASET_PATH,
    "fasterrcnn_beach_human_clean.pth"
)

ORIGINAL_MODEL = os.path.join(
    DATASET_PATH,
    "fasterrcnn_beach_human.pth"
)

# ============================================================
# 2. MODEL
# ============================================================

if os.path.exists(CLEAN_MODEL):
    MODEL_PATH = CLEAN_MODEL
else:
    MODEL_PATH = ORIGINAL_MODEL

print("Model:")
print(MODEL_PATH)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

NUM_CLASSES = 3

model = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(
    weights=None
)

in_features = (
    model.roi_heads
    .box_predictor
    .cls_score
    .in_features
)

model.roi_heads.box_predictor = FastRCNNPredictor(
    in_features,
    NUM_CLASSES
)

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device
)

model.load_state_dict(checkpoint)

model.to(device)
model.eval()

print("✅ Model loaded successfully!")

# ============================================================
# 3. SETTINGS
# ============================================================

CONFIDENCE_THRESHOLD = 0.50
IOU_THRESHOLD = 0.50

MODEL_TO_RAW = {
    1: 0,   # model class 1 -> land
    2: 1    # model class 2 -> water
}

# ============================================================
# 4. YOUR REQUESTED IMAGES
# ============================================================

requested_images = [
    "1640154603-Wed-Dec-22_16_30_03-AEST-2021-BURLEIGH-TWR-snap-4Gtrailcam_jpg.rf.JRKPYKB0XT4cxKdpZm3b.jpg",

    "1640150102-Wed-Dec-22_15_15_02-AEST-2021-BURLEIGH-TWR-snap-4Gtrailcam_jpg.rf.WSuIM7YlMnrO43HvoTSi.jpg",

    "1640151902-Wed-Dec-22_15_45_02-AEST-2021-BURLEIGH-TWR-snap-4Gtrailcam_jpg.rf.3WehESpX0Qzw7Q91a9NU.jpg",

    "1640149202-Wed-Dec-22_15_00_02-AEST-2021-BURLEIGH-TWR-snap-4Gtrailcam_jpg.rf.ls85n6g3Mrg5KoE9rfKw.jpg",

    "1640155505-Wed-Dec-22_16_45_05-AEST-2021-BURLEIGH-TWR-snap-4Gtrailcam_jpg.rf.osRyOFMWqcUQgxgfWbLo.jpg",

    "1640155503-Wed-Dec-22_16_45_03-AEST-2021-BURLEIGH-TWR-snap-4Gtrailcam_jpg.rf.622aUMZx7BgnBQE6c4Y5.jpg",

    "1640135702-Wed-Dec-22_11_15_02-AEST-2021-BURLEIGH-TWR-snap-4Gtrailcam_jpg.rf.7e47Phitp5U5GjN16rdq.jpg",

    "1640217602-Thu-Dec-23_10_00_02-AEST-2021-BURLEIGH-TWR-snap-4Gtrailcam_jpg.rf.1y4GGYLPw1jTry0zY4fQ.jpg",

    "1639953906-Mon-Dec-20_08_45_06-AEST-2021-BURLEIGH-TWR-snap-4Gtrailcam_jpg.rf.0dGyCV5KVtDrcRPFVPhs.jpg",

    "1640293205-Fri-Dec-24_07_00_05-AEST-2021-BURLEIGH-TWR-snap-4Gtrailcam_jpg.rf.ZaOpUQjD4cHVJMEGcoiO.jpg"
]

# ============================================================
# 5. FIND AVAILABLE REQUESTED IMAGES
# ============================================================

available = []

print("\n" + "=" * 70)
print("CHECKING REQUESTED IMAGES")
print("=" * 70)

for filename in requested_images:

    path = os.path.join(
        TEST_IMAGE_DIR,
        filename
    )

    if os.path.exists(path):

        available.append(path)

        print("✅", filename)

    else:

        print("❌ Missing:", filename)

print(
    "\nRequested images available:",
    len(available),
    "/ 10"
)

# ============================================================
# 6. AUTOMATICALLY FILL MISSING SLOTS
# ============================================================

all_test_images = sorted([
    os.path.join(TEST_IMAGE_DIR, f)
    for f in os.listdir(TEST_IMAGE_DIR)
    if f.lower().endswith(
        (".jpg", ".jpeg", ".png", ".bmp")
    )
])

used_names = set(
    os.path.basename(p)
    for p in available
)

remaining_images = [
    p for p in all_test_images
    if os.path.basename(p) not in used_names
]

needed = 10 - len(available)

if needed > 0:

    replacements = remaining_images[:needed]

    print("\n" + "=" * 70)
    print("REPLACING MISSING IMAGES")
    print("=" * 70)

    for p in replacements:

        print(
            "🔄 Replacement:",
            os.path.basename(p)
        )

        available.append(p)

# Make sure exactly 10
image_paths = available[:10]

print("\n" + "=" * 70)
print("FINAL 10 IMAGES")
print("=" * 70)

for i, p in enumerate(image_paths, 1):

    print(
        f"{i}. {os.path.basename(p)}"
    )

print(
    "\nTotal selected:",
    len(image_paths)
)

if len(image_paths) < 10:

    raise RuntimeError(
        "The test folder contains fewer than 10 images."
    )

# ============================================================
# 7. READ GROUND TRUTH LABELS
# ============================================================

def read_ground_truth(image_path):

    image_name = os.path.basename(
        image_path
    )

    label_name = (
        os.path.splitext(image_name)[0]
        + ".txt"
    )

    label_path = os.path.join(
        TEST_LABEL_DIR,
        label_name
    )

    if not os.path.exists(label_path):

        print(
            "⚠️ No label found:",
            label_name
        )

        return [], []

    image = Image.open(
        image_path
    ).convert("RGB")

    width, height = image.size

    boxes = []
    labels = []

    with open(label_path, "r") as f:

        for line in f:

            parts = line.strip().split()

            if len(parts) != 5:
                continue

            cls = int(parts[0])

            if cls not in [0, 1]:
                continue

            xc = float(parts[1])
            yc = float(parts[2])
            w = float(parts[3])
            h = float(parts[4])

            x1 = (xc - w / 2) * width
            y1 = (yc - h / 2) * height

            x2 = (xc + w / 2) * width
            y2 = (yc + h / 2) * height

            boxes.append([
                x1, y1, x2, y2
            ])

            labels.append(cls)

    return boxes, labels


# ============================================================
# 8. IOU
# ============================================================

def calculate_iou(box1, box2):

    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])

    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection = (
        max(0, x2 - x1) *
        max(0, y2 - y1)
    )

    area1 = (
        max(0, box1[2] - box1[0]) *
        max(0, box1[3] - box1[1])
    )

    area2 = (
        max(0, box2[2] - box2[0]) *
        max(0, box2[3] - box2[1])
    )

    union = area1 + area2 - intersection

    if union <= 0:
        return 0

    return intersection / union


# ============================================================
# 9. RUN ALL 10 IMAGES
# ============================================================

results = []

all_gt_labels = []
all_pred_labels = []

total_tp = 0
total_fp = 0
total_fn = 0

print("\n")
print("=" * 70)
print("RUNNING MODEL ON 10 IMAGES")
print("=" * 70)

for image_number, image_path in enumerate(
    image_paths,
    start=1
):

    image = Image.open(
        image_path
    ).convert("RGB")

    image_tensor = F.to_tensor(
        image
    ).to(device)

    # --------------------------------------------------------
    # PREDICTION
    # --------------------------------------------------------

    with torch.no_grad():

        prediction = model(
            [image_tensor]
        )[0]

    boxes = prediction["boxes"].cpu().numpy()
    labels = prediction["labels"].cpu().numpy()
    scores = prediction["scores"].cpu().numpy()

    keep = scores >= CONFIDENCE_THRESHOLD

    boxes = boxes[keep]
    labels = labels[keep]
    scores = scores[keep]

    raw_labels = np.array([
        MODEL_TO_RAW.get(
            int(label),
            int(label)
        )
        for label in labels
    ])

    # --------------------------------------------------------
    # PREDICTED COUNTS
    # --------------------------------------------------------

    pred_land = int(
        np.sum(raw_labels == 0)
    )

    pred_water = int(
        np.sum(raw_labels == 1)
    )

    pred_total = (
        pred_land +
        pred_water
    )

    # --------------------------------------------------------
    # GROUND TRUTH
    # --------------------------------------------------------

    gt_boxes, gt_labels = read_ground_truth(
        image_path
    )

    gt_land = sum(
        1 for x in gt_labels
        if x == 0
    )

    gt_water = sum(
        1 for x in gt_labels
        if x == 1
    )

    gt_total = len(gt_labels)

    # --------------------------------------------------------
    # MATCH PREDICTIONS TO GROUND TRUTH
    # --------------------------------------------------------

    matched_gt = set()

    tp = 0
    fp = 0

    for pred_box, pred_label in zip(
        boxes,
        raw_labels
    ):

        best_iou = 0
        best_gt = -1

        for j, gt_box in enumerate(
            gt_boxes
        ):

            if j in matched_gt:
                continue

            iou = calculate_iou(
                pred_box,
                gt_box
            )

            if iou > best_iou:

                best_iou = iou
                best_gt = j

        if best_iou >= IOU_THRESHOLD:

            matched_gt.add(best_gt)

            tp += 1

            all_gt_labels.append(
                gt_labels[best_gt]
            )

            all_pred_labels.append(
                int(pred_label)
            )

        else:

            fp += 1

    fn = len(gt_boxes) - len(matched_gt)

    total_tp += tp
    total_fp += fp
    total_fn += fn

    # --------------------------------------------------------
    # DISPLAY IMAGE
    # --------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(13, 8)
    )

    ax.imshow(image)

    for box, raw_label, score in zip(
        boxes,
        raw_labels,
        scores
    ):

        x1, y1, x2, y2 = box

        rect = patches.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            linewidth=2,
            edgecolor="lime",
            facecolor="none"
        )

        ax.add_patch(rect)

        ax.text(
            x1,
            max(y1 - 5, 5),
            f"{raw_label}:{score:.2f}",
            color="lime",
            fontsize=9,
            weight="bold"
        )

    # --------------------------------------------------------
    # COUNT PANEL
    # --------------------------------------------------------

    panel_text = (
        f"Land (0): {pred_land}    "
        f"Water (1): {pred_water}    "
        f"Total: {pred_total}\n"
        f"Ground Truth: {gt_total}"
    )

    ax.text(
        0.02,
        0.98,
        panel_text,
        transform=ax.transAxes,
        fontsize=11,
        weight="bold",
        color="white",
        verticalalignment="top",
        bbox=dict(
            boxstyle="round,pad=0.35",
            facecolor="black",
            alpha=0.65,
            edgecolor="none"
        )
    )

    ax.set_title(
        f"Image {image_number}/10",
        fontsize=13,
        weight="bold"
    )

    ax.axis("off")

    plt.tight_layout()

    plt.show()

    # --------------------------------------------------------
    # PRINT
    # --------------------------------------------------------

    print(
        f"{image_number}. "
        f"{os.path.basename(image_path)}"
    )

    print(
        f"   Prediction → "
        f"Land={pred_land}, "
        f"Water={pred_water}, "
        f"Total={pred_total}"
    )

    print(
        f"   Ground Truth → "
        f"Land={gt_land}, "
        f"Water={gt_water}, "
        f"Total={gt_total}"
    )

    print(
        f"   TP={tp}, FP={fp}, FN={fn}"
    )

    print()

    results.append({

        "Image": image_number,

        "Filename":
            os.path.basename(image_path),

        "Ground Truth Land":
            gt_land,

        "Predicted Land":
            pred_land,

        "Ground Truth Water":
            gt_water,

        "Predicted Water":
            pred_water,

        "Ground Truth Total":
            gt_total,

        "Predicted Total":
            pred_total,

        "Error":
            pred_total - gt_total,

        "Absolute Error":
            abs(pred_total - gt_total),

        "TP":
            tp,

        "FP":
            fp,

        "FN":
            fn
    })


# ============================================================
# 10. RESULTS TABLE
# ============================================================

df = pd.DataFrame(results)

print("\n")
print("=" * 90)
print("FINAL RESULTS FOR 10 IMAGES")
print("=" * 90)

display(df)

# ============================================================
# 11. OVERALL METRICS
# ============================================================

precision_det = (
    total_tp /
    (total_tp + total_fp)
    if total_tp + total_fp > 0
    else 0
)

recall_det = (
    total_tp /
    (total_tp + total_fn)
    if total_tp + total_fn > 0
    else 0
)

f1_det = (
    2 * precision_det * recall_det /
    (precision_det + recall_det)
    if precision_det + recall_det > 0
    else 0
)

print("\n")
print("=" * 60)
print("OVERALL DETECTION METRICS")
print("=" * 60)

print("TP =", total_tp)
print("FP =", total_fp)
print("FN =", total_fn)

print(
    f"Precision = {precision_det:.4f}"
)

print(
    f"Recall    = {recall_det:.4f}"
)

print(
    f"F1 Score  = {f1_det:.4f}"
)

# ============================================================
# 12. COUNTING METRICS
# ============================================================

print("\n")
print("=" * 60)
print("COUNTING PERFORMANCE")
print("=" * 60)

print(
    "Total Ground Truth:",
    df["Ground Truth Total"].sum()
)

print(
    "Total Predicted:",
    df["Predicted Total"].sum()
)

print(
    "Mean Absolute Error:",
    round(
        df["Absolute Error"].mean(),
        3
    )
)

print(
    "Total Absolute Error:",
    df["Absolute Error"].sum()
)

# ============================================================
# 13. CONFUSION MATRIX
# ============================================================

if len(all_gt_labels) > 0:

    y_true = np.array(
        all_gt_labels
    )

    y_pred = np.array(
        all_pred_labels
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    print("\n")
    print("=" * 60)
    print("CONFUSION MATRIX")
    print("=" * 60)

    print(cm)

    fig, ax = plt.subplots(
        figsize=(7, 6)
    )

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=[
            "0: Land",
            "1: Water"
        ]
    )

    disp.plot(
        ax=ax,
        cmap="Blues"
    )

    plt.title(
        "Confusion Matrix — 10 Images"
    )

    plt.tight_layout()

    plt.show()

    # --------------------------------------------------------
    # CLASSIFICATION METRICS
    # --------------------------------------------------------

    print(
        "Classification Precision:",
        round(
            precision_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),
            4
        )
    )

    print(
        "Classification Recall:",
        round(
            recall_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),
            4
        )
    )

    print(
        "Classification F1:",
        round(
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),
            4
        )
    )

# ============================================================
# 14. GRAPH — GROUND TRUTH VS PREDICTED TOTAL
# ============================================================

x = np.arange(
    1,
    len(df) + 1
)

plt.figure(
    figsize=(12, 6)
)

plt.plot(
    x,
    df["Ground Truth Total"],
    marker="o",
    linewidth=2,
    label="Ground Truth"
)

plt.plot(
    x,
    df["Predicted Total"],
    marker="s",
    linewidth=2,
    label="Predicted"
)

plt.xlabel("Image Number")
plt.ylabel("Number of People")

plt.title(
    "Ground Truth vs Predicted People Count"
)

plt.xticks(x)

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()

plt.show()

# ============================================================
# 15. GRAPH — LAND
# ============================================================

plt.figure(
    figsize=(12, 6)
)

plt.plot(
    x,
    df["Ground Truth Land"],
    marker="o",
    linewidth=2,
    label="Ground Truth Land (0)"
)

plt.plot(
    x,
    df["Predicted Land"],
    marker="s",
    linewidth=2,
    label="Predicted Land (0)"
)

plt.xlabel("Image Number")
plt.ylabel("People")

plt.title(
    "Land People — Ground Truth vs Prediction"
)

plt.xticks(x)

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()

plt.show()

# ============================================================
# 16. GRAPH — WATER
# ============================================================

plt.figure(
    figsize=(12, 6)
)

plt.plot(
    x,
    df["Ground Truth Water"],
    marker="o",
    linewidth=2,
    label="Ground Truth Water (1)"
)

plt.plot(
    x,
    df["Predicted Water"],
    marker="s",
    linewidth=2,
    label="Predicted Water (1)"
)

plt.xlabel("Image Number")
plt.ylabel("People")

plt.title(
    "Water People — Ground Truth vs Prediction"
)

plt.xticks(x)

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()

plt.show()

# ============================================================
# 17. GRAPH — COUNT ERROR
# ============================================================

plt.figure(
    figsize=(12, 6)
)

plt.bar(
    x,
    df["Error"]
)

plt.axhline(
    0,
    linewidth=1
)

plt.xlabel("Image Number")
plt.ylabel("Prediction Error")

plt.title(
    "People Count Error"
)

plt.xticks(x)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.show()

# ============================================================
# 18. GRAPH — TP / FP / FN
# ============================================================

plt.figure(
    figsize=(12, 6)
)

plt.plot(
    x,
    df["TP"],
    marker="o",
    linewidth=2,
    label="TP"
)

plt.plot(
    x,
    df["FP"],
    marker="s",
    linewidth=2,
    label="FP"
)

plt.plot(
    x,
    df["FN"],
    marker="^",
    linewidth=2,
    label="FN"
)

plt.xlabel("Image Number")
plt.ylabel("Detections")

plt.title(
    "TP / FP / FN per Image"
)

plt.xticks(x)

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()

plt.show()

# ============================================================
# 19. SAVE RESULTS
# ============================================================

RESULTS_DIR = os.path.join(
    DATASET_PATH,
    "results"
)

os.makedirs(
    RESULTS_DIR,
    exist_ok=True
)

csv_path = os.path.join(
    RESULTS_DIR,
    "10_images_results.csv"
)

df.to_csv(
    csv_path,
    index=False
)

print("\n")
print("=" * 70)
print("✅ COMPLETE")
print("=" * 70)

print(
    "Results CSV saved at:"
)

print(csv_path)

print("\nThe 10 images were successfully evaluated.")